# Курсова робота: Аналіз продажів і комерційної діяльності підприємства
**Дисципліна:** Мови програмування для аналізу даних  
**Спеціальність:** 122 «Комп’ютерні науки»

## 1. Постановка завдання та дослідницькі питання
Мета дослідження — провести повний цикл дослідницького та статистичного аналізу даних роздрібної мережі **Sample Superstore** засобами Python (Pandas, NumPy, Matplotlib, Seaborn) для виявлення факторів прибутковості та формування практичних бізнес-рекомендацій.

**Дослідницькі питання:**
1. Які категорії та підкатегорії товарів генерують найбільший прибуток, а які є збитковими?
2. Як розмір наданої знижки впливає на прибутковість і маржинальність замовлень?
3. Чим відрізняються комерційні показники в розрізі регіонів (та регіональних менеджерів) і окремих штатів?
4. Які відмінності існують між сегментами клієнтів та способами доставки?
5. Чи існують статистичні взаємозв'язки та аномальні спостереження (викиди) серед фінансових показників?


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='Set2', font='DejaVu Sans')

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../results/figures')
EXPORTS_DIR = Path('../results/exports')


## 2. Завантаження та первинне дослідження даних
Завантажуємо основну таблицю SampleSuperstore.csv та два пов'язані довідники: 
egional_managers.csv (регіональні менеджери й хаби) та shipping_modes.csv (параметри доставки).


In [ ]:
sales_df = pd.read_csv(RAW_DIR / 'SampleSuperstore.csv')
managers_df = pd.read_csv(RAW_DIR / 'regional_managers.csv')
shipping_df = pd.read_csv(RAW_DIR / 'shipping_modes.csv')

print('Розмір основної таблиці:', sales_df.shape)
display(sales_df.head())
sales_df.info()


In [ ]:
# Описова статистика та кількість унікальних значень
display(sales_df.describe().round(2))
print('\nКількість унікальних значень за ознаками:')
print(sales_df.nunique())


## 3. Попереднє опрацювання та очищення даних
Перевіряємо пропущені значення, дублікати, типи даних, некоректні значення та потенційні викиди за методом IQR.


In [ ]:
print('Пропущені значення:\n', sales_df.isna().sum())
print('\nКількість повних дублікатів:', sales_df.duplicated().sum())

# Видалення 17 повних дублікатів
df = sales_df.drop_duplicates().copy()
print(f'Рядків до очищення: {len(sales_df)}, після очищення: {len(df)}')

# Приведення Postal Code до 5-значного рядкового формату з ведучими нулями
df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)

# Перевірка логічних обмежень
print('Sales <= 0:', (df['Sales'] <= 0).sum())
print('Quantity <= 0:', (df['Quantity'] <= 0).sum())
print('Discount поза [0, 1]:', ((df['Discount'] < 0) | (df['Discount'] > 1)).sum())


In [ ]:
# Оцінка потенційних викидів за методом IQR (без сліпого видалення, оскільки це реальні великі/збиткові замовлення)
for col in ['Sales', 'Profit', 'Quantity']:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    cnt = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f'{col}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, викидів={cnt} ({cnt/len(df)*100:.2f}%)')


## 4. Об'єднання пов'язаних таблиць і створення нових аналітичних ознак
Виконуємо два обґрунтовані об'єднання (merge) та створюємо 6 нових похідних ознак для поглибленого аналізу.


In [ ]:
# Об'єднання №1: додавання регіональних менеджерів і хабів
df = df.merge(managers_df, on='Region', how='left')

# Об'єднання №2: додавання SLA та базової вартості доставки
df = df.merge(shipping_df, on='Ship Mode', how='left')

# Нові аналітичні ознаки
df['Profit_Margin_Pct'] = np.round((df['Profit'] / df['Sales']) * 100, 2)
df['Unit_Price'] = np.round(df['Sales'] / df['Quantity'], 2)
df['Estimated_Cost'] = np.round(df['Sales'] - df['Profit'], 2)
df['Profit_Status'] = np.where(df['Profit'] > 0, 'Прибуткове', np.where(df['Profit'] < 0, 'Збиткове', 'Беззбиткове'))
df['Order_Size'] = pd.cut(
    df['Sales'],
    bins=[0, 50, 200, 1000, np.inf],
    labels=['Малий (до \)', 'Середній (\–200)', 'Великий (\–1000)', 'Преміум (>\)']
)
df['Discount_Tier'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0.0, 0.2, 0.4, 1.0],
    labels=['Без знижки (0%)', 'Помірна (1–20%)', 'Середня (21–40%)', 'Висока (>40%)']
)

print('Кількість ознак після збагачення:', df.shape[1])
df.to_csv(PROCESSED_DIR / 'superstore_cleaned.csv', index=False, encoding='utf-8-sig')
display(df.head(3))


## 5. Статистичний та аналітичний аналіз (Pandas)
Виконуємо групування, агрегування, ранжування, побудову зведеної таблиці (pivot_table), таблиці спряженості (crosstab) та кореляційний аналіз.


In [ ]:
# 5.1. Агрегування за категоріями товарів
cat_summary = df.groupby('Category').agg(
    Orders=('Sales', 'count'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', 'mean')
).round(2)
cat_summary['Profit_Margin_Pct'] = (cat_summary['Total_Profit'] / cat_summary['Total_Sales'] * 100).round(2)
display(cat_summary.sort_values('Total_Profit', ascending=False))

# 5.2. Ранжування підкатегорій за прибутком
sub_summary = df.groupby(['Category', 'Sub-Category']).agg(
    Orders=('Sales', 'count'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', 'mean')
).round(2)
sub_summary['Rank'] = sub_summary['Total_Profit'].rank(ascending=False).astype(int)
display(sub_summary.sort_values('Total_Profit', ascending=False))


In [ ]:
# 5.3. Аналіз за регіонами та регіональними менеджерами
reg_summary = df.groupby(['Region', 'Regional_Manager']).agg(
    Orders=('Sales', 'count'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', 'mean')
).round(2)
reg_summary['Profit_Margin_Pct'] = (reg_summary['Total_Profit'] / reg_summary['Total_Sales'] * 100).round(2)
display(reg_summary.sort_values('Total_Profit', ascending=False))

# 5.4. Вплив рівня знижки на прибуток
disc_summary = df.groupby('Discount_Tier', observed=False).agg(
    Orders=('Sales', 'count'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Margin_Pct=('Profit_Margin_Pct', 'mean')
).round(2)
display(disc_summary)


In [ ]:
# 5.5. Зведена таблиця (pivot_table): Регіон x Категорія (сумарний прибуток)
pivot_profit = pd.pivot_table(
    df, values='Profit', index='Region', columns='Category',
    aggfunc='sum', margins=True, margins_name='Разом'
).round(2)
display(pivot_profit)

# 5.6. Таблиця спряженості (crosstab): Сегмент x Статус прибутковості (%)
display(pd.crosstab(df['Segment'], df['Profit_Status'], normalize='index').mul(100).round(2))

# 5.7. Кореляційна матриця
display(df[['Sales', 'Quantity', 'Discount', 'Profit', 'Profit_Margin_Pct', 'Unit_Price']].corr().round(3))


## 6. Візуалізація даних (Matplotlib та Seaborn)
Будуємо 7 змістовних графіків, що відповідають дослідницьким питанням (результати також автоматично зберігаються у 
esults/figures/ скриптом src/run_analysis.py).


In [ ]:
# 6.1. Розподіл обсягів продажів (log10) та прибутку
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(np.log10(df['Sales']), bins=35, kde=True, color='#2b7bba', ax=axes[0])
axes[0].set_title('Розподіл обсягів продажів (log10)')
axes[0].set_xlabel('log10(Продажі, USD)')
axes[0].set_ylabel('Кількість замовлень')

sns.histplot(df['Profit'].clip(-500, 500), bins=40, kde=True, color='#2ca02c', ax=axes[1])
axes[1].axvline(0, color='#d62728', linestyle='--', linewidth=1.5)
axes[1].set_title('Розподіл прибутку (- ... +)')
axes[1].set_xlabel('Прибуток, USD')
axes[1].set_ylabel('Кількість замовлень')
plt.tight_layout()
plt.show()


In [ ]:
# 6.2. Рейтинг підкатегорій за сумарним прибутком
sub_agg = df.groupby('Sub-Category')['Profit'].sum().sort_values()
colors = ['#ef4444' if v < 0 else '#10b981' for v in sub_agg.values]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(sub_agg.index, sub_agg.values, color=colors)
ax.axvline(0, color='#333333', linewidth=1)
ax.set_title('Рейтинг підкатегорій товарів за сумарним прибутком')
ax.set_xlabel('Сумарний прибуток, USD')
plt.tight_layout()
plt.show()


In [ ]:
# 6.3. Вплив знижки на прибуток та середню маржинальність
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.55, s=30, ax=axes[0])
axes[0].axhline(0, color='#d62728', linestyle='--')
axes[0].set_title('Залежність прибутку від знижки')

disc_avg = df.groupby('Discount_Tier', observed=False)['Profit_Margin_Pct'].mean().reset_index()
sns.barplot(data=disc_avg, x='Discount_Tier', y='Profit_Margin_Pct', ax=axes[1])
axes[1].axhline(0, color='#333333')
axes[1].set_title('Середня маржинальність за рівнем знижки')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# 6.4. Boxplot маржинальності та теплова карта зведеної таблиці
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.boxplot(data=df, x='Category', y='Profit_Margin_Pct', hue='Segment', ax=axes[0])
axes[0].axhline(0, color='#d62728', linestyle='--')
axes[0].set_title('Розподіл маржинальності (%) за категоріями та сегментами')

pivot_df = pd.pivot_table(df, values='Profit', index='Region', columns='Category', aggfunc='sum')
sns.heatmap(pivot_df, annot=True, fmt=',.0f', cmap='YlGn', linewidths=0.6, ax=axes[1])
axes[1].set_title('Сумарний прибуток (USD): Регіон × Категорія')
plt.tight_layout()
plt.show()


## 7. Основні висновки
1. **Лідери та збиткові напрямки:** Категорія Technology забезпечує 36.41% виручки (\.2 тис.) та найбільший прибуток (\.5 тис., маржа 17.40%). Категорія Furniture попри \.3 тис. продажів приносить лише \.4 тис. прибутку (маржа 2.49%) через збитковість підкатегорій Tables (-\.7 тис.) та Bookcases (-\.5 тис.).
2. **Вплив знижок:** Між розміром знижки та маржинальністю виявлено дуже сильний обернений зв'язок (
 = -0.864). Знижки понад 20% системно переводять замовлення у збиток (сумарний збиток понад \.4 тис.).
3. **Регіональна диференціація:** Найефективнішим регіоном є West (менеджер Anna Andreadi, маржа 14.94%), найслабшим — Central (менеджер Kelly Williams, маржа 7.92%) через високу середню знижку (24.03%) і збитки в Техасі та Іллінойсі.
